# Q5 — Duration Analysis

How long do COVID-19 clinical trials take?  
How do completed vs stopped trials differ in duration? Where are the outliers?

---

## Setup

In [45]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sqlalchemy import create_engine

# ── Setup ──────────────────────────────────────────────────────────────────
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sqlalchemy import create_engine, text

# ─────────────────────────────────────────
# CONFIG — update username if needed
# ─────────────────────────────────────────
DB_USER = "vittoriobariosco"  # your username
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "clinical_trials" # your PostgreSQL database name 
CSV_PATH = "/Users/vittoriobariosco/Documents/APPLICATIONS/MIGx/data/processed/clinical_trials_clean.csv"  #path to cleaned CSV from EDA notebook

# ─────────────────────────────────────────
# 1. CONNECT TO POSTGRESQL
# ─────────────────────────────────────────
engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

# --- Style Guide ---
COLORS = {
    'primary': '#2563EB',
    'success': '#10B981',
    'danger': '#EF4444',
    'accent': '#F59E0B',
    'secondary': '#64748B',
    'light_gray': '#F1F5F9',
    'dark': '#1E293B'
}

PALETTE = [COLORS['primary'], COLORS['success'], COLORS['danger'],
           COLORS['accent'], COLORS['secondary'], '#8B5CF6', '#EC4899', '#06B6D4']

def style_fig(fig, title='', height=500, width=None, showlegend=True):
    fig.update_layout(
        title=dict(text=title, font=dict(family='Helvetica Neue', size=18, color=COLORS['dark']),
                   x=0, xanchor='left', pad=dict(l=10, t=10)),
        font=dict(family='Helvetica Neue', size=12, color=COLORS['dark']),
        plot_bgcolor='white',
        paper_bgcolor='white',
        height=height,
        width=width,
        showlegend=showlegend,
        margin=dict(l=60, r=30, t=60, b=50),
        legend=dict(bgcolor='rgba(0,0,0,0)', font=dict(size=11))
    )
    fig.update_xaxes(showgrid=False, linecolor='#E2E8F0', linewidth=1)
    fig.update_yaxes(showgrid=True, gridcolor='#F1F5F9', linecolor='#E2E8F0', linewidth=1)
    return fig

print('Setup OK — connected to PostgreSQL')

Setup OK — connected to PostgreSQL


---
## Query 5A — Duration Distribution (Completed Trials)


In [46]:
query_5a = """
WITH durations AS (
    SELECT
        study_id,
        phase,
        study_type,
        (completion_date - start_date) AS duration_days

    FROM studies
    WHERE
        status_group = 'Completed'     
        AND completion_date IS NOT NULL  
        AND start_date IS NOT NULL
        AND pre_covid_start = FALSE      
        AND date_error = FALSE           
        AND (completion_date - start_date) >= 0
)

SELECT
    phase,
    COUNT(*) AS n_trials,

    ROUND(PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY duration_days)::numeric, 0) AS duration_q1,
    ROUND(PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY duration_days)::numeric, 0) AS duration_median,
    ROUND(PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY duration_days)::numeric, 0) AS duration_q3,

    ROUND(AVG(duration_days)::numeric, 0) AS duration_mean,

    MIN(duration_days)::int AS duration_min,
    MAX(duration_days)::int AS duration_max

FROM durations
WHERE phase != 'Unknown'
GROUP BY phase
ORDER BY duration_median;
"""

df_5a = pd.read_sql(query_5a, engine)
df_5a

,phase,n_trials,duration_q1,duration_median,duration_q3,duration_mean,duration_min,duration_max
0,Phase 1,35,53.0,91.0,135.0,103.0,8,225
1,Not Applicable,215,55.0,93.0,157.0,114.0,0,424
2,Phase 4,22,61.0,132.0,237.0,147.0,28,339
3,Phase 3,75,93.0,138.0,189.0,147.0,19,335
4,Phase 2,93,88.0,150.0,213.0,152.0,14,354
5,Early Phase 1,2,109.0,158.0,208.0,158.0,59,257


In [18]:
# 1. Prepare and sort the data by median duration
df_plot = df_5a.sort_values('duration_median', ascending=True)

# 1. Define the logical clinical order
phase_order = ['Early Phase 1', 'Phase 1', 'Phase 2', 'Phase 3', 'Phase 4']

# 2. Filter and set Categorical order
df_filtered = df_5a[df_5a['phase'].isin(phase_order)].copy()
df_filtered['phase'] = pd.Categorical(df_filtered['phase'], categories=phase_order, ordered=True)

# 3. Sort: Ascending=False puts 'Early Phase 1' at the end of the DF, 
# which Plotly renders at the TOP of the Y-axis.
df_plot = df_filtered.sort_values('phase', ascending=False)

# Create descriptive labels for the Y-axis
df_plot['y_label'] = df_plot.apply(lambda r: f"{r['phase']} (n={int(r['n_trials'])})", axis=1)

fig = go.Figure()

# --- Trace 1: Median Bars ---
# We remove the 'text' from here to prevent overlapping
fig.add_trace(go.Bar(
    y=df_plot['y_label'],
    x=df_plot['duration_median'],
    orientation='h',
    marker_color=COLORS['primary'],
    name='Median Duration',
    hovertemplate="<b>%{y}</b><br>Median: %{x} days<extra></extra>"
))

# --- Trace 2: IQR Error Bars ---
fig.add_trace(go.Scatter(
    y=df_plot['y_label'],
    x=df_plot['duration_median'],
    error_x=dict(
        type='data',
        symmetric=False,
        array=(df_plot['duration_q3'] - df_plot['duration_median']).tolist(),
        arrayminus=(df_plot['duration_median'] - df_plot['duration_q1']).tolist(),
        color=COLORS['secondary'],
        thickness=2,
        width=8
    ),
    mode='markers',
    marker=dict(size=10, color=COLORS['accent'], symbol='diamond'),
    name='IQR (Q1–Q3)',
    hovertemplate="<b>IQR Range</b><br>Q1: %{customdata[0]} d<br>Q3: %{customdata[1]} d<extra></extra>",
    customdata=df_plot[['duration_q1', 'duration_q3']]
))

# --- Step 4: Add Offset Annotations (The "Total Day" labels) ---
# We loop through the data to place text exactly above the bar segments
for i, row in df_plot.iterrows():
    fig.add_annotation(
        x=row['duration_median'],
        y=row['y_label'],
        text=f"<b>{row['duration_median']:,.0f} d</b>",
        showarrow=False,
        yshift=18, 
        xshift=15,            # Moves the text 18 pixels ABOVE the bar
        xanchor='center',
        font=dict(size=11, color=COLORS['dark'])
    )

# Final Styling
style_fig(fig, title='<b> Median Trial Duration by Phase</b>', height=500)
fig.update_xaxes(title_text='Total Days (Start to Completion)', showgrid=True, gridcolor='#F1F5F9')
fig.update_layout(
    showlegend=True,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    margin=dict(l=200, r=50, t=100, b=50)
)

fig.show()

* **Efficiency of Early Safety**: **Phase 1** trials achieved the fastest completion speed with a median of **91 days** (~3 months).
* **The Efficacy Window**: Core research phases (**Phase 2 and 3**) show a consistent duration plateau between **138 and 150 days**.
* **Operational Variability**: **Phase 4** studies show the highest variability, with IQRs spanning over 100 days. This reflects the diverse nature of long-term monitoring compared to the standardized, rapid-response structure of interventional phases.
* **Statistical Note**: While **Early Phase 1** shows the highest median (**158 days**), this is based on a very small sample size ($n=2$) and should be viewed as an outlier rather than a broad trend.

In [47]:
query_5b = """
WITH durations AS (
    SELECT
        s.study_id,
        (s.completion_date - s.start_date) AS duration_days
    FROM studies s
    WHERE
        s.status_group = 'Completed'
        AND s.completion_date IS NOT NULL
        AND s.start_date IS NOT NULL
        AND s.pre_covid_start = FALSE
        AND s.date_error = FALSE
        AND (s.completion_date - s.start_date) >= 0
),
study_area_map AS (
    SELECT DISTINCT
        study_id,
        therapeutic_area
    FROM conditions
    WHERE therapeutic_area IS NOT NULL
)

SELECT
    sam.therapeutic_area,
    COUNT(d.study_id) AS n_trials,


    ROUND(PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY d.duration_days)::numeric, 0) AS duration_q1,
    ROUND(PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY d.duration_days)::numeric, 0) AS duration_median,
    ROUND(PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY d.duration_days)::numeric, 0) AS duration_q3,


    ROUND(AVG(d.duration_days)::numeric, 0) AS duration_mean,

    MIN(d.duration_days)::int AS duration_min,
    MAX(d.duration_days)::int AS duration_max

FROM durations d
JOIN study_area_map sam ON d.study_id = sam.study_id
GROUP BY sam.therapeutic_area

HAVING COUNT(*) >= 10

ORDER BY duration_median DESC;
"""

df_5b = pd.read_sql(query_5b, engine)
df_5b

,therapeutic_area,n_trials,duration_q1,duration_median,duration_q3,duration_mean,duration_min,duration_max
0,Infectious Disease,801,54.0,92.0,168.0,117.0,0,390
1,Metabolic/Cardiovascular,55,48.0,91.0,195.0,123.0,7,355
2,Respiratory/Critical Care,115,48.0,84.0,180.0,112.0,0,355
3,Other,294,42.0,83.0,150.0,104.0,0,424
4,Mental Health,89,33.0,75.0,150.0,103.0,7,424
5,Musculoskeletal/Rheumatology,17,47.0,62.0,91.0,84.0,7,304
6,Oncology,15,48.0,61.0,180.0,117.0,21,390


In [48]:
# 1. Prepare and sort data by median duration (Slowest to Fastest)
df_plot_area = df_5b.sort_values('duration_median', ascending=True)

# Create labels with sample size (n) for statistical context
df_plot_area['y_label'] = df_plot_area.apply(
    lambda r: f"{r['therapeutic_area']} (n={int(r['n_trials'])})", axis=1
)

fig = go.Figure()

# --- Trace 1: Median Bars ---
fig.add_trace(go.Bar(
    y=df_plot_area['y_label'],
    x=df_plot_area['duration_median'],
    orientation='h',
    marker_color=COLORS['primary'],
    name='Median Duration',
    hovertemplate="<b>%{y}</b><br>Median: %{x} days<extra></extra>"
))

# --- Trace 2: IQR Error Bars ---
fig.add_trace(go.Scatter(
    y=df_plot_area['y_label'],
    x=df_plot_area['duration_median'],
    error_x=dict(
        type='data',
        symmetric=False,
        array=(df_plot_area['duration_q3'] - df_plot_area['duration_median']).tolist(),
        arrayminus=(df_plot_area['duration_median'] - df_plot_area['duration_q1']).tolist(),
        color=COLORS['secondary'],
        thickness=2,
        width=8
    ),
    mode='markers',
    marker=dict(size=10, color=COLORS['accent'], symbol='diamond'),
    name='IQR (Q1–Q3)',
    hovertemplate="<b>IQR Range</b><br>Q1: %{customdata[0]} d<br>Q3: %{customdata[1]} d<extra></extra>",
    customdata=df_plot_area[['duration_q1', 'duration_q3']]
))

# --- Step 3: Add Offset Annotations ---
for i, row in df_plot_area.iterrows():
    fig.add_annotation(
        x=row['duration_median'],
        y=row['y_label'],
        text=f"<b>{row['duration_median']:,.0f} d</b>",
        showarrow=False,
        yshift=18, 
        xshift=15,
        xanchor='center',
        font=dict(size=11, color=COLORS['dark'])
    )

# Final Styling
style_fig(fig, title='<b> Median Trial Duration by Therapeutic Area</b>', height=550)
fig.update_xaxes(title_text='Total Days (Start to Completion)')
fig.update_layout(
    margin=dict(l=220, r=50, t=100, b=50),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)

fig.show()

### Terapeutical Area Analysis
---

* **Infectious Disease** ($n=801$) establishes the primary performance standard with a median duration of **92 days**. 
* **Respiratory/Critical Care** (84 days) and **Mental Health** (75 days) significantly outperformed the volume benchmark.
* **Metabolic Variance**: While its median (**91 days**) aligns with the benchmark, the **Metabolic/Cardiovascular** sector exhibits a massive "long tail," with its **Q3 at 195 days**.
* **Small-Sample Outliers**: **Oncology** and **Musculoskeletal** report the fastest turnarounds (~61 days).
* **Skewed Distributions**: In every sector, the **Mean is higher than the Median**, confirming that a small subset of "troubled" or complex trials significantly inflates the overall average duration across the board.

In [ ]:
query_5c = """
WITH durations AS (
    SELECT
        s.study_id,
        s.phase,
        (s.completion_date - s.start_date) AS duration_days
    FROM studies s
    WHERE
        s.status_group = 'Completed'
        AND s.completion_date IS NOT NULL
        AND s.start_date IS NOT NULL
        AND s.pre_covid_start = FALSE
        AND s.date_error = FALSE
        AND (s.completion_date - s.start_date) >= 0
        AND s.phase NOT IN ('Unknown', 'Not Applicable')
),
study_area_map AS (
    SELECT DISTINCT
        study_id,
        therapeutic_area
    FROM conditions
    WHERE therapeutic_area IS NOT NULL
)

SELECT
    sam.therapeutic_area,
    d.phase,
    COUNT(*) AS n_trials,
    ROUND(PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY d.duration_days)::numeric, 0) AS duration_median,
    ROUND(PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY d.duration_days)::numeric, 0) AS duration_q1,
    ROUND(PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY d.duration_days)::numeric, 0) AS duration_q3

FROM durations d
JOIN study_area_map sam ON d.study_id = sam.study_id
GROUP BY sam.therapeutic_area, d.phase
HAVING COUNT(*) >= 1
ORDER BY sam.therapeutic_area, d.phase;
"""

df_5c = pd.read_sql(query_5c, engine)
df_5c

,therapeutic_area,phase,n_trials,duration_median,duration_q1,duration_q3
0,Infectious Disease,Early Phase 1,2,158.0,109.0,208.0
1,Infectious Disease,Phase 1,27,115.0,62.0,150.0
2,Infectious Disease,Phase 2,92,148.0,87.0,214.0
3,Infectious Disease,Phase 3,72,137.0,92.0,187.0
4,Infectious Disease,Phase 4,22,132.0,61.0,237.0
5,Mental Health,Phase 3,1,111.0,111.0,111.0
6,Metabolic/Cardiovascular,Phase 1,1,195.0,195.0,195.0
7,Metabolic/Cardiovascular,Phase 2,1,17.0,17.0,17.0
8,Metabolic/Cardiovascular,Phase 4,1,194.0,194.0,194.0
9,Oncology,Phase 1,1,60.0,60.0,60.0


In [44]:
import plotly.express as px

# 1. Sort the phase order logically
phase_order = ['Early Phase 1', 'Phase 1', 'Phase 2', 'Phase 3', 'Phase 4']
df_5c['phase'] = pd.Categorical(df_5c['phase'], categories=phase_order, ordered=True)

# 2. Create the Facet Plot
fig = px.bar(
    df_5c.sort_values(['therapeutic_area', 'phase']),
    x='duration_median',
    y='phase',
    facet_col='therapeutic_area',
    facet_col_wrap=3,      # 3 mini-charts per row
    color='therapeutic_area',
    color_discrete_sequence=px.colors.qualitative.Safe,
    orientation='h',
    text='duration_median',
    labels={'duration_median': 'Median Days', 'phase': 'Clinical Phase'},
    title='<b>The Clinical Pipeline by Sector: Median Duration (Days)</b>',
    hover_data=['n_trials', 'duration_q1', 'duration_q3']
)

# 3. Enhance formatting
fig.update_traces(texttemplate='%{text} d', textposition='outside')
fig.update_traces(cliponaxis=False)
fig.update_layout(
    height=800, 
    showlegend=False,
    margin=dict(t=100, l=120)
)

# Remove "therapeutic_area=" from subplot titles for a cleaner look
fig.for_each_annotation(lambda a: a.update(text=f"<b>{a.text.split('=')[-1]}</b>"))

fig.show()

### Summary: Clinical Trial Duration Benchmarks**

The typical duration for a COVID-19 related clinical trial in this dataset ranges from **3 to 5 months (90–150 days)**.
---

### **1. Duration by Clinical Phase**
Research follows a predictable "U-shaped" curve of efficiency:
* **Safety (Phase 1):** The most agile phase with a median of **91 days**. 
* **The "Efficacy Squeeze" (Phases 2 & 3):** These phases represent the longest investment, peaking at a median of **150 days for Phase 2** and **138 days for Phase 3**. The increased complexity of patient recruitment and endpoint monitoring accounts for this ~5-month window.
* **Post-Market (Phase 4):** Shows a median of **132 days**, though with the highest variance (Q3 of 237 days), indicating that once a drug is on the market, monitoring timelines vary wildly based on the specific symptom being tracked.

### **2. Duration by Therapeutic Area**
* **The Volume Standard:** **Infectious Disease** ($n=801$) sets the global pace with a median of **92 days**.
* **Sector Leaders:** **Mental Health (75 days)** and **Respiratory (84 days)** proved more agile than the average.
* **Metabolic/Cardiovascular** trials, while having a similar median (91 days), show a high **Q3 of 195 days**.

---

### **3. Significant Outliers: What Takes Longer Than Expected?**

The dashboard and cross-tabulated data highlight specific "bottlenecks" where trials significantly exceed the expected median:

* **Respiratory Phase 2:** At **164 days**, these trials take nearly double the duration of the general Respiratory median (84 days).
* **Metabolic Phase 1 & 4:** Both show durations near **195 days**. This is an extreme deviation from the Phase 1 benchmark of 91 days.
* **Infectious Disease "Early Phase 1":** While Phase 1 is fast, "Early Phase 1" takes **158 days**.
